In [1]:
import pandas as pd


def load_pgoH_design(filepath):
    """
    Load DOE design file with columns:
        sim_id, tol, Tcrit, V, mu, kk1, kk2, kappa, k1
    """
    cols = ["sim_id", "tol", "Tcrit", "V", "mu", "kk1", "kk2", "kappa", "k1"]

    df = pd.read_csv(
        filepath,
        delim_whitespace=True,
        header=None,
        names=cols
    )

    df["sim_id"] = df["sim_id"].astype(int)
    return df


def filter_design(
    df,
    high=None,
    low=None,
    high_q=None,
    low_q=None,
    sort_by=None,
    ascending=False
):
    """
    Filter DOE rows by absolute thresholds and/or quantile-based thresholds.

    Parameters
    ----------
    df : pd.DataFrame
        Design dataframe.
    high : dict or None
        Absolute lower bounds, e.g. {"mu": 0.20, "k1": 8.0}
        keeps rows with df[col] >= value
    low : dict or None
        Absolute upper bounds, e.g. {"kappa": 0.05}
        keeps rows with df[col] <= value
    high_q : dict or None
        Quantile-based "high" filters, e.g. {"mu": 0.8}
        keeps rows with df[col] >= df[col].quantile(0.8)
    low_q : dict or None
        Quantile-based "low" filters, e.g. {"kappa": 0.2}
        keeps rows with df[col] <= df[col].quantile(0.2)
    sort_by : str, list[str], or None
        Column(s) to sort output by.
    ascending : bool or list[bool]
        Sort direction.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe.
    """
    out = df.copy()

    if high is not None:
        for col, val in high.items():
            out = out[out[col] >= val]

    if low is not None:
        for col, val in low.items():
            out = out[out[col] <= val]

    if high_q is not None:
        for col, q in high_q.items():
            thresh = df[col].quantile(q)
            out = out[out[col] >= thresh]

    if low_q is not None:
        for col, q in low_q.items():
            thresh = df[col].quantile(q)
            out = out[out[col] <= thresh]

    if sort_by is not None:
        out = out.sort_values(by=sort_by, ascending=ascending)

    return out.reset_index(drop=True)

In [2]:
filepath = "PGOH_Ideal_tol_Tcrit_V_mu_kk1_kk2_kappa_k1_design.txt"
df = load_pgoH_design(filepath)

print(df.head())

   sim_id     tol  Tcrit           V     mu    kk1    kk2  kappa     k1
0       1  18.688  1.048  695735.362  0.214  1.860  1.019  0.318  4.468
1       2  25.772  1.105  580848.556  0.148  2.282  0.573  0.036  5.943
2       3  20.185  1.047  535445.172  0.132  1.547  0.500  0.222  7.743
3       4  19.863  1.103  236359.061  0.111  1.517  1.142  0.258  4.552
4       5  17.694  1.108  266870.122  0.092  2.001  0.457  0.025  3.246


C:\Users\jml2414\AppData\Local\Temp\ipykernel_32312\1807117017.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


In [29]:
subset = filter_design(
    df,
    high_q={"V": 0.8, "tol": 0.8},
    sort_by=["V", "tol"],
    ascending=[True, True]
)

print(subset)
print(subset["sim_id"].tolist())

    sim_id     tol  Tcrit           V     mu    kk1    kk2  kappa     k1
0      318  29.799  1.055  581609.982  0.106  1.043  0.506  0.125  5.475
1      780  27.074  1.015  585573.440  0.136  1.479  1.161  0.012  6.660
2      476  29.200  1.153  590708.098  0.180  1.121  1.074  0.107  4.671
3      394  28.165  1.107  596862.790  0.213  2.292  0.599  0.224  3.961
4      570  27.937  1.106  601548.304  0.137  1.766  0.414  0.306  5.212
5      195  28.399  1.124  606432.013  0.221  1.303  0.688  0.104  5.458
6      932  29.834  1.039  607871.881  0.142  1.591  0.478  0.017  3.829
7       92  28.686  1.041  609241.206  0.214  1.881  0.701  0.317  9.661
8      803  29.240  1.173  617542.008  0.193  0.987  0.434  0.272  7.686
9      536  27.366  1.142  620528.266  0.186  1.579  0.663  0.231  7.786
10     768  28.393  1.127  621828.487  0.130  1.636  0.697  0.275  4.303
11      11  29.255  1.180  622626.800  0.140  2.062  0.980  0.185  6.686
12     282  28.956  1.097  626351.816  0.204  1.028

In [9]:
print(df["Tcrit"].max())

1.2


## Flip Fluid Elems

In [1]:
import numpy as np

# Load the text file
data = np.loadtxt("Fluid_elems_needs_flipped.txt", delimiter=",")
# Make a copy
data_flipped = data.copy()

# Swap columns 3 and 5 (1-based indexing -> columns 2 and 4 in Python)
data_flipped[:, [2, 4]] = data_flipped[:, [4, 2]]

# Save result if desired
np.savetxt("Fluid_elems_flipped.txt", data_flipped, delimiter=",", fmt="%g")